# Prox-Pruning nnU-Net: полный пайплайн (final)

**Структурный прунинг nnU-Net через proximal optimization с Knowledge Distillation.**
Задача: сегментация 31 структуры головы и шеи на КТ. Архитектура: ResEncUNet-M (101.6M параметров).

## Этапы

| Этап | Ячейки | Что происходит |
|---|---|---|
| **A. Prox-finetune + KD** | 1–11 | Конвергированная модель дообучается с prox-регуляризатором (group-lasso): слабые каналы «умирают». Учитель — 5-фолдовый nnU-Net-ансамбль (KD) |
| **B. Физическая резка** | 12–14 | Каналы с нулевой нормой вырезаются из свёрток |
| **C. Recovery-KD** | 15–17 | Короткое KD-дообучение резаной модели |
| **D. Валидация + INT8** | 18–20 | Sliding-window валидация, NNCF INT8, argmax-гейт |

## Результаты (H&N, 31 класс, ResEncUNet-M)

| Модель | Kaggle 15-case val | Локальные 10 кейсов | Время/кейс (CPU) |
|---|---|---|---|
| Дистиллят (baseline) | 0.8800 | 0.8394 (INT8) | 5.9 мин |
| Prox (−36.5% каналов) | **0.916** | 0.8214 (INT8) | 5.3 мин |
| Prox + резка + recovery | 0.8435 | 0.7918 (INT8) | ~5.5 мин |

*0.916 включает эффект попутной второй дистилляции (+100 эп KD).*

## Ключевые находки

1. **Momentum — главный переключатель спарсификации.** При m=0.99 окно λ_g×Σlr×N
   не существует. При m=0.9 окно открывается при λ_g=1e-3.
2. **Post-hoc prox режет только глубокие/FLOPs-дешёвые стадии** (enc5 100%, enc4 −30%,
   ранние FLOPs-тяжёлые нетронуты: −36.5% каналов = −4.2% FLOPs). Противоположно
   Slim nnU-Net — для резки ранних стадий нужен prox **с эпохи 0**.
3. **SpinalCord не выживает физическую резку** — потерян полностью на обеих шкалах.
4. **KD при prox-finetune = попутная вторая дистилляция.*

## Источники

| Источник | Тип | Содержимое |
|---|---|---|
| `iliaszalialov/nnunet-distill` | ядро | nnUNet_preprocessed + дистиллят-чекпоинт |
| `alenasnigireva/hn-results` | датасет | 5-фолдовый учитель |
| `alenasnigireva/prox-ckpt` | датасет | чекпоинты этапа A (resume) |

**Переменные — см. следующую ячейку с полными комментариями.**


In [ ]:
# ═══════════════ КОНФИГУРАЦИЯ ═══════════════
# Каждая переменная с расшифровкой и влиянием ↑/↓

import os, multiprocessing as mp, time
mp.set_start_method('spawn', force=True)
os.environ['nnUNet_raw']          = '/kaggle/working/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/kaggle/working/nnUNet_preprocessed'
os.environ['nnUNet_results']      = '/kaggle/working/nnUNet_results'
for d in ('nnUNet_raw', 'nnUNet_results'):
    os.makedirs('/kaggle/working/' + d, exist_ok=True)
KERNEL_T0 = time.time()

# ─── ДАННЫЕ И ПУТИ ────────────────────────────────────────────────
DATASET_ID = 1
# ID nnU-Net датасета (определяется автоматически из имени папки).

CONFIG = '3d_fullres'
# Конфигурация nnU-Net: 3D патчи 56×192×192.

FOLD = 0
# Фолд для обучения/валидации (эксперимент вёлся на fold 0).

# ─── PROX-РЕГУЛЯРИЗАЦИЯ (этап A: спарсификация) ──────────────────
GROUP_LAMBDA = 1e-3
# λ_g — сила group-lasso. ГЛАВНАЯ ручка спарсификации.
# Prox-шаг вычитает λ_g×lr из нормы КАЖДОГО канала на каждой итерации.
#   ↑ (1e-2…5e-2): быстрее умирают, глубже спарсенность, риск коллапса Dice
#   ↓ (3e-4): безопаснее, но может не начаться вовсе
# Проверено при m=0.9: 1e-3 → −36.5% каналов за 70 prox-эпох.

INDIV_LAMBDA = 1e-2
# λ₁ — ℓ1 на отдельные веса (soft-threshold внутри канала).
#   ↑: быстрее умирают слабые веса, риск для мелких структур
#   ↓ (0): только group-компонента
# Соотношение λ₁ : λ_g ≈ 10:1 — из Slim nnU-Net (prox-mix).

WARMUP_EPOCHS = 30
# Эпохи чистого KD до включения prox (current_epoch <= WARMUP → prox выключен).
#   ↑ (50): сеть лучше освоит задачу до резки
#   ↓ (0): prox с первого шага — рискованно на converged-весах

NUM_EPOCHS = 100
# Горизонт prox-finetune. Определяет кумулятивный шринк:
#   Σ_shrink ≈ λ_g × Σ_epoch(lr_e) × 250_итеров
#   ↑: глубже спарсенность   ↓: мельче резка
# При m=0.9 и λ=1e-3: 100 эп → −36.5% каналов.

# ─── ОПТИМИЗАТОР ──────────────────────────────────────────────────
LR_INIT = 1e-4
# Начальный learning rate (PolyLR от этого значения до 0).
#   ↑ (3e-4): быстрее, риск коллапса редких классов
#   ↓ (3e-5): безопаснее, медленнее

MOMENTUM = 0.9
# Momentum SGD. КЛЮЧЕВАЯ ПЕРЕМЕННАЯ: определяет баланс
# «prox-вычет ↔ градиентное восстановление» норм каналов.
#   m=0.99 → шаг ×100 → восстановление подавляет прунинг (окна НЕТ)
#   m=0.9  → шаг ×10  → окно открывается при λ_g=1e-3
#   ↑ к 0.99: подавляет спарсификацию.   ↓ к 0.8: ускоряет, шумнее.

WEIGHT_DECAY = 3e-5
# L2-регуляризация оптимизатора (стандарт nnU-Net).
# Не путать с prox-шагом: wd — гладкое, prox — резкий структурный вычет.

RECOVERY_LR = 1e-4
# lr для recovery-этапа (после физической резки, 40 эп KD).

RECOVERY_MOMENTUM = 0.9
# Momentum recovery (аналогично — 0.9, не 0.99).

RECOVERY_EPOCHS = 40
# Эпохи recovery-KD после физической резки.
#   ↑ (100): глубже, но SpinalCord не вернуть и за 100
#   ↓ (20): мельче
# 40 эп достаточно для стабилизации EMA.

# ─── ФИЗИЧЕСКАЯ РЕЗКА (этап B) ────────────────────────────────────
ALIVE_REL_THRESHOLD = 1e-2
# Относительный порог живости: ‖Θ_g‖ > max_стадии × threshold.
#   ↑ (5e-2): глубже резка, выше риск потери качества
#   ↓ (1e-3): консервативнее
# Значение 1e-2 согласовано с prox (sparsity_log).

ALIVE_ABS_THRESHOLD = 1e-4
# Абсолютный порог (защита от нулевых стадий). Канал жив при ОБЕИХ условиях.

MIN_CHANNELS = {'stem': 20, 'enc0': 20, 'enc1': 28, 'enc2': 40,
                'enc3': 80, 'enc4': 96, 'enc5': 96}
# Минимум каналов на стадию при резке (защита от вырезания в ноль).

# ─── ПРОЧЕЕ ───────────────────────────────────────────────────────
BUDGET_H = 9.5
# Мягкий бюджет сессии (часов). Чистый стоп с сохранением чекпоинта.
# Kaggle-лимит: 12 ч GPU / 30 ч неделя. Оставлять ≥ 2 ч запаса.

SMOKE = True
# True  — смоук: 2 эп × 5 итераций (~15 мин), проверка пайплайна
# False — полный прогон (prox 100 эп + резка + recovery 40 эп ≈ 17 ч = 2 сессии)

RESUME = False
# True — продолжить с checkpoint_latest (датасет-посредник prox-ckpt).

SEED = 42
# Зерно случайности.

NUM_DL_WORKERS = 2
# Воркеры dataloader'а (2 — профилактика дедлоков на Kaggle 4 vCPU).

print('Конфигурация задана.')
print('  λ_g=%g | λ₁=%g | m=%g | lr=%g' % (GROUP_LAMBDA, INDIV_LAMBDA, MOMENTUM, LR_INIT))
print('  warmup=%d | epochs=%d | recovery=%d эп | SMOKE=%s' % (
    WARMUP_EPOCHS, NUM_EPOCHS, RECOVERY_EPOCHS, SMOKE))

In [ ]:
# ============ ЯЧЕЙКА 2: ДАННЫЕ + ЧЕКПОИНТ + УЧИТЕЛЬ ============
import subprocess, sys, glob, json, os, copy, gc, random, shutil, time
def _pip(*a): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=True)
try:
    import nnunetv2
except ImportError:
    _pip('nnunetv2')

# lasso/proximal (prox-шаг этапа A) — из репозитория nnunet-sparse (Ido-zh);
# в публичный репо не мерджится, только клон в рабочую папку сессии
REPO_SPARSE = '/kaggle/working/nnunet-sparse'
if not os.path.exists(REPO_SPARSE):
    os.system('git clone https://github.com/Ido-zh/nnunet-sparse ' + REPO_SPARSE)
sys.path.insert(0, REPO_SPARSE)
for _sub in ('lasso', 'trainer'):
    assert os.path.isdir(os.path.join(REPO_SPARSE, _sub)), f'нет папки {_sub}/ в клоне nnunet-sparse'

# зависимости этапа D (ONNX + NNCF INT8); пины — как в проверенном quantization-final
sys.modules.setdefault('tensorflow', None)   # NNCF иначе пытается импортировать TF
sys.modules.setdefault('tensorboard', None)
for _pkg, _imp in (('nncf==2.19.0', 'nncf'), ('openvino==2025.2.0', 'openvino'),
                   ('onnx', 'onnx'), ('onnxscript', 'onnxscript')):
    try:
        __import__(_imp)
    except ImportError:
        print('pip install', _pkg)
        _pip(_pkg)

print('маунты /kaggle/input:', os.listdir('/kaggle/input'))
import torch
import numpy as np
import random

def _rc(path):
    m, c = path.rsplit('.', 1)
    return getattr(__import__(m, fromlist=[c]), c)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

from batchgenerators.utilities.file_and_folder_operations import join, load_json, save_json
from dynamic_network_architectures.architectures.unet import ResidualEncoderUNet

# preprocessed данные (из ядра nnunet-distill)
pp = sorted(glob.glob('/kaggle/input/**/nnUNet_preprocessed/Dataset*_*', recursive=True))
pp = [h for h in pp if glob.glob(os.path.join(h, '*', '*.b2nd'))]
assert pp, 'nnUNet_preprocessed не найден'
pre_dir = pp[0]
DATASET_NAME = os.path.basename(pre_dir)
DATASET_ID = int(DATASET_NAME.split('_')[0].replace('Dataset', ''))
plans = load_json(join(pre_dir, 'nnUNetResEncUNetMPlans.json'))
plans.setdefault('continue_training', False)  # ключ обязателен: nnUNetTrainer.__init__ делает plans.pop('continue_training')
dataset_json = load_json(join(pre_dir, 'dataset.json'))
cfg = plans['configurations'][CONFIG]
kw = dict(cfg['architecture']['arch_kwargs'])
FEATS_BASE = list(kw['features_per_stage'])
PATCH = tuple(cfg['patch_size'])
IN_CH = len(dataset_json['channel_names'])
NUM_CLASSES = len(dataset_json['labels'])
print('датасет:', DATASET_NAME, '| база:', FEATS_BASE)

# дистиллят-чекпоинт (стартовые веса)
dc = sorted(glob.glob('/kaggle/input/**/nnUNetTrainerDistill__*/fold_0/checkpoint_final.pth', recursive=True))
assert dc, 'дистиллят не найден'
ckpt_distill = torch.load(dc[0], map_location='cpu', weights_only=False)
print('дистиллят: эп.', ckpt_distill.get('current_epoch'))

# учитель: 5-фолдовый ансамбль
th = sorted(glob.glob('/kaggle/input/**/nnUNetTrainerNoMirroring__nnUNetResEncUNetMPlans__3d_fullres', recursive=True))
th = [h for h in th if all(os.path.isfile(os.path.join(h, 'fold_%d' % i, 'checkpoint_final.pth')) for i in range(5))]
assert th, '5-фолдовый учитель не найден'
TEACHER_DIR = th[0]
print('учитель:', TEACHER_DIR)

# рабочая копия preprocessed
ds_work_root = '/kaggle/working/nnUNet_preprocessed'
ds_work = join(ds_work_root, DATASET_NAME)
data_id = cfg.get('data_identifier', 'nnUNetPlans_3d_fullres')
real_data = join(pre_dir, data_id)
if not os.path.isdir(real_data):
    real_data = [d for d in glob.glob(join(pre_dir, '*_3d_fullres')) if os.path.isdir(d)][0]
if not os.path.exists(ds_work):
    os.makedirs(ds_work)
    for fn in ('dataset.json', 'dataset_fingerprint.json', 'splits_final.json', 'nnUNetResEncUNetMPlans.json'):
        shutil.copy(join(pre_dir, fn), join(ds_work, fn))
    if os.path.isdir(join(pre_dir, 'gt_segmentations')):
        shutil.copytree(join(pre_dir, 'gt_segmentations'), join(ds_work, 'gt_segmentations'))
if not os.path.exists(join(ds_work, data_id)):
    os.symlink(real_data, join(ds_work, data_id))
os.environ['nnUNet_preprocessed'] = ds_work_root

def _rc(path):
    m, c = path.rsplit('.', 1)
    return getattr(__import__(m, fromlist=[c]), c)

def build_net(feats6):
    return ResidualEncoderUNet(
        input_channels=IN_CH, num_classes=NUM_CLASSES,
        n_stages=kw['n_stages'], features_per_stage=list(feats6),
        conv_op=_rc(kw['conv_op']), kernel_sizes=kw['kernel_sizes'],
        strides=kw['strides'], n_blocks_per_stage=kw['n_blocks_per_stage'],
        n_conv_per_stage_decoder=kw['n_conv_per_stage_decoder'],
        conv_bias=kw['conv_bias'], norm_op=_rc(kw['norm_op']),
        norm_op_kwargs=kw['norm_op_kwargs'], dropout_op=None,
        dropout_op_kwargs=None, nonlin=_rc(kw['nonlin']),
        nonlin_kwargs=kw['nonlin_kwargs'])

BASE = build_net(FEATS_BASE)
BASE.load_state_dict(ckpt_distill['network_weights'], strict=True)
BASE.eval()
print('BASE: %.1fM параметров' % (sum(p.numel() for p in BASE.parameters())/1e6))

In [ ]:
# ============ ЯЧЕЙКА 3: МЕХАНИКА РЕЗКИ (План C v12) ============
# Проверена в 5 прогонах. Слайсит encoder, decoder-зеркало, transpconvs, seg_layers.
import copy
import torch.nn as _nn

CHAIN = ['stem', 'enc0', 'enc1', 'enc2', 'enc3', 'enc4', 'enc5']

def get_blocks(container):
    for attr in ('blocks', 'convs'):
        if hasattr(container, attr):
            return list(getattr(container, attr))
    raise TypeError(f'{type(container).__name__}: нет блоков')

def _stage_container(net, st):
    return net.encoder.stem if st == 'stem' else net.encoder.stages[int(st[3:])]

def _units_of(block):
    out = []
    for tag in ('conv1', 'conv2', 'conv'):
        b = getattr(block, tag, None)
        if b is None:
            continue
        conv = getattr(b, 'conv', b)
        if isinstance(conv, _nn.modules.conv._ConvNd):
            out.append((tag, conv, getattr(b, 'norm', None)))
    sk = getattr(block, 'skip', None)
    if sk is not None and isinstance(sk, _nn.Module):
        for sub in sk:
            conv = getattr(sub, 'conv', sub)
            if isinstance(conv, _nn.modules.conv._ConvNd):
                out.append(('skip', conv, getattr(sub, 'norm', None)))
    assert out, f'{type(block).__name__}: нет свёрток'
    return out

if 'DECODER_CAT_SKIP_FIRST' not in globals():
    def _detect_cat_order():
        import inspect, importlib, pkgutil
        import dynamic_network_architectures as dna
        for mod_name in ('dynamic_network_architectures.building_blocks.simple_decoders',
                         'dynamic_network_architectures.building_blocks.unet',
                         'dynamic_network_architectures.architectures.unet'):
            try:
                m = importlib.import_module(mod_name)
                if hasattr(m, 'UNetDecoder'):
                    break
            except ModuleNotFoundError:
                continue
        if m is None:
            for _, name, _i in pkgutil.walk_packages(dna.__path__, dna.__name__ + '.'):
                try:
                    m = importlib.import_module(name)
                except Exception:
                    continue
                if hasattr(m, 'UNetDecoder'):
                    break
        assert m is not None, 'UNetDecoder не найден'
        src = inspect.getsource(m.UNetDecoder.forward)
        for line in src.splitlines():
            s = line.strip()
            if '.cat(' in s or 'torch.cat' in s:
                inner = s[s.find('('):]
                return not inner.strip('( [').startswith('x')
        raise RuntimeError('torch.cat не найден')
    DECODER_CAT_SKIP_FIRST = _detect_cat_order()
    print('DECODER_CAT_SKIP_FIRST =', DECODER_CAT_SKIP_FIRST)

def _slice_out(conv, keep, norm=None):
    k = keep.to(conv.weight.device)
    conv.weight.data = conv.weight.data.index_select(0, k).contiguous()
    if conv.bias is not None:
        conv.bias.data = conv.bias.data.index_select(0, k).contiguous()
    conv.out_channels = len(k)
    if norm is not None and getattr(norm, 'weight', None) is not None:
        norm.weight.data = norm.weight.data.index_select(0, k).contiguous()
        norm.bias.data = norm.bias.data.index_select(0, k).contiguous()
    if norm is not None and getattr(norm, 'num_features', None) is not None:
        norm.num_features = len(k)

def _slice_in(conv, keep):
    k = keep.to(conv.weight.device)
    conv.weight.data = conv.weight.data.index_select(1, k).contiguous()
    conv.in_channels = len(k)

def _force_norm_sync(block, n_out, keep=None):
    for name, m in block.named_modules():
        if isinstance(m, (_nn.InstanceNorm1d, _nn.InstanceNorm2d, _nn.InstanceNorm3d,
                          _nn.BatchNorm1d, _nn.BatchNorm2d, _nn.BatchNorm3d)):
            if m.weight is not None and m.weight.shape[0] != n_out:
                if keep is not None:
                    k = keep.to(m.weight.device)
                    m.weight.data = m.weight.data.index_select(0, k).contiguous()
                    if m.bias is not None:
                        m.bias.data = m.bias.data.index_select(0, k).contiguous()
                m.num_features = n_out

def _normalize_skips(enc, CHAIN):
    changed = 0
    for st in CHAIN:
        cont = enc.stem if st == 'stem' else enc.stages[int(st[3:])]
        blocks = list(cont.convs) if st == 'stem' else get_blocks(cont)
        for b in blocks:
            sk = getattr(b, 'skip', None)
            if sk is None or not isinstance(sk, _nn.Module):
                continue
            c1 = getattr(b, 'conv1', None) or getattr(b, 'conv', None)
            cs = getattr(getattr(c1, 'conv', c1), 'stride', (1,1,1)) if c1 is not None else (1,1,1)
            conv1x1 = None
            for sub in sk:
                cv = getattr(sub, 'conv', sub)
                if isinstance(cv, _nn.modules.conv._ConvNd):
                    conv1x1 = cv
                    break
            if conv1x1 is not None and conv1x1.in_channels == conv1x1.out_channels:
                avg = None
                for sub in sk:
                    if isinstance(sub, _nn.AvgPool3d):
                        avg = sub
                        break
                if avg is not None:
                    b.skip = _nn.Sequential(_nn.AvgPool3d(kernel_size=avg.kernel_size,
                                                          stride=avg.stride, padding=avg.padding))
                elif all(s == 1 for s in cs):
                    b.skip = (lambda x: x)
                else:
                    b.skip = _nn.Sequential(_nn.AvgPool3d(kernel_size=cs, stride=cs, padding=0))
                changed += 1
    if changed:
        print(f'  нормализация skip: удалено {changed} избыточных skip-conv')
    return changed

def _sync_duplicate_encoder(net, fK, CHAIN):
    enc_dup = getattr(net.decoder, 'encoder', None)
    if enc_dup is None:
        return
    named_src = dict(net.encoder.named_modules())
    src_params = dict(net.encoder.named_parameters())
    for name, md_ in enc_dup.named_modules():
        ms = named_src.get(name)
        if isinstance(ms, _nn.modules.conv._ConvNd) and isinstance(md_, _nn.modules.conv._ConvNd):
            md_.in_channels = ms.in_channels
            md_.out_channels = ms.out_channels
            md_.weight = _nn.Parameter(ms.weight.data.clone())
            if ms.bias is not None:
                md_.bias = _nn.Parameter(ms.bias.data.clone())
    for name, p_src in src_params.items():
        parts = name.split('.')
        mod = enc_dup
        try:
            for pth in parts[:-1]:
                mod = getattr(mod, pth)
            if getattr(mod, parts[-1], None) is not None and \
               getattr(mod, parts[-1]).shape != p_src.shape:
                setattr(mod, parts[-1], _nn.Parameter(p_src.data.clone()))
        except AttributeError:
            continue
    for st in CHAIN:
        keep = fK.get(st)
        if keep is None:
            continue
        cont = enc_dup.stem if st == 'stem' else enc_dup.stages[int(st[3:])]
        blocks = list(cont.convs) if st == 'stem' else get_blocks(cont)
        for b in blocks:
            _force_norm_sync(b, len(keep), keep)
    print('  decoder.encoder синхронизирован')

def fix_monotonic(fp6, strides):
    fp6 = list(fp6)
    for i in range(1, len(fp6)):
        m = int(strides[i][0] * strides[i][1] * strides[i][2])
        fp6[i] = max(int(min(fp6[i], fp6[i-1] * m)), 8)
    return fp6

In [ ]:
# ============ ЯЧЕЙКА 4: СПАРС-ЛОГ + KD-КОМПОНЕНТЫ ============
import csv
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer, collate_outputs
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels

def group_sparsity_universal(model, rel=ALIVE_REL_THRESHOLD, abs_=ALIVE_ABS_THRESHOLD):
    out = {}
    with torch.no_grad():
        for name, m in model.named_modules():
            if isinstance(m, (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Conv3d)):
                gd = 0
            elif isinstance(m, (torch.nn.ConvTranspose1d, torch.nn.ConvTranspose2d,
                                torch.nn.ConvTranspose3d)):
                gd = 1
            else:
                continue
            if m.groups != 1:
                continue
            W = m.weight.transpose(0, gd).reshape(m.weight.shape[gd], -1)
            norms = W.norm(dim=1)
            alive = (norms > norms.max() * rel) & (norms > abs_)
            out[name] = alive
    return out

def _stage_key(name):
    parts = name.split('.')
    if parts[0] == 'encoder' and parts[1] == 'stem':
        return 'enc_stem'
    if parts[0] == 'encoder' and len(parts) > 2:
        return 'enc_' + parts[2]
    if parts[0] == 'decoder' and parts[1] == 'stages':
        return 'dec_' + parts[2]
    if parts[0] == 'decoder':
        return 'dec_' + parts[1]
    return parts[0]

pm = PlansManager(plans)
cm = pm.get_configuration(CONFIG)
lm = pm.get_label_manager(dataset_json)
num_in = determine_num_input_channels(pm, cm, dataset_json)

TEACHER_DEVICE = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
TEACHERS = []
for fold_i in range(5):
    sd_t = torch.load(join(TEACHER_DIR, 'fold_%d' % fold_i, 'checkpoint_final.pth'),
                      map_location='cpu', weights_only=False)
    tnet = nnUNetTrainer.build_network_architecture(pm, cm, num_in,
                                                    lm.num_segmentation_heads, True)
    tnet.load_state_dict(sd_t['network_weights'])
    tnet.eval().to(TEACHER_DEVICE)
    tnet.half()
    for p_ in tnet.parameters():
        p_.requires_grad_(False)
    TEACHERS.append(tnet)
    del sd_t
    print('учитель fold %d: OK' % fold_i)
torch.cuda.empty_cache()

@torch.no_grad()
def teacher_probs_fn(data):
    d = data.to(TEACHER_DEVICE, non_blocking=True).half()
    acc = None
    for net_t in TEACHERS:
        logits = net_t(d)
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
        p = torch.softmax(logits.float(), dim=1)
        acc = p if acc is None else acc + p
    return (acc / len(TEACHERS)).detach().to(data.device)

def soft_ce(sl, tp):
    logp = torch.log_softmax(sl.float(), dim=1)
    return -(tp * logp).sum(dim=1).mean()

def soft_dice(sl, tp, eps=1e-6):
    s = torch.softmax(sl.float(), dim=1)
    dims = (0, 2, 3, 4)
    inter = (s * tp).sum(dims)
    denom = s.sum(dims) + tp.sum(dims)
    return (1.0 - (2.0 * inter + eps) / (denom + eps)).mean()

class _SessionOver(Exception):
    pass

In [ ]:
# ============ ЯЧЕЙКА 5: PROX-ТРЕНЕР (этап A) ============
# KD-лосс + prox-шаг. prox_step из lasso.proximal (Ido-zh/nnunet-sparse).
from lasso.proximal import prox_step_unet_group_sparsity_

class nnUNetTrainerProxKD(nnUNetTrainer):
    def __init__(self, plans, configuration, fold, dataset_json,
                 device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.ft_warmup = WARMUP_EPOCHS
        self.save_every = 5
        self.num_epochs = NUM_EPOCHS
        self.num_val_iterations_per_epoch = 10
        self._t0 = None
        self.kd_teachers = TEACHERS
        self.kd_teacher_device = TEACHER_DEVICE

    def configure_optimizers(self, lr_init=None, new_weight_decay=None):
        optimizer = torch.optim.SGD(self.network.parameters(), LR_INIT,
                                    weight_decay=WEIGHT_DECAY,
                                    momentum=MOMENTUM, nesterov=True)
        scheduler = torch.optim.lr_scheduler.PolynomialLR(
            optimizer, total_iters=self.num_epochs, power=1.0)
        self.optimizer, self.lr_scheduler = optimizer, scheduler
        return optimizer, scheduler

    def train_step(self, batch):
        data = batch['data'].to(self.device, non_blocking=True)
        target = batch['target']
        if isinstance(target, list):
            target = [t.to(self.device, non_blocking=True) for t in target]
        else:
            target = target.to(self.device, non_blocking=True)
        self.optimizer.zero_grad(set_to_none=True)
        with torch.autocast(self.device.type, enabled=self.device.type == 'cuda'):
            output = self.network(data)
            shapes = [tuple(o.shape[2:]) for o in output]
            tp = teacher_probs_fn(data)
            t_list = []
            for shp in shapes:
                if tuple(tp.shape[2:]) == tuple(shp):
                    t_list.append(tp)
                else:
                    t_list.append(torch.nn.functional.interpolate(
                        tp, size=shp, mode='area').clamp_min(1e-6))
            n = len(output)
            ws = [1.0 / (2 ** i) for i in range(n)]
            ws = [w / sum(ws) for w in ws]
            loss = sum(w * (soft_ce(o, t) + soft_dice(o, t))
                       for w, o, t in zip(ws, output, t_list))
        if self.grad_scaler is not None:
            self.grad_scaler.scale(loss).backward()
            self.grad_scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.grad_scaler.step(self.optimizer)
            self.grad_scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.optimizer.step()
        if self.current_epoch > self.ft_warmup:
            lr = self.optimizer.param_groups[0]['lr']
            prox_step_unet_group_sparsity_(
                self.network, lr=lr,
                indiv_lambda=INDIV_LAMBDA,
                group_lambda=GROUP_LAMBDA,
                mode='out',
                apply_individual_l1=True,
                include_bias_l1=True,
                include_norm_layer=False)
        return {'loss': loss.detach().cpu().numpy()}

    def on_train_epoch_end(self, train_outputs):
        outputs = collate_outputs(train_outputs)
        loss_here = np.mean(outputs['loss'])
        self.logger.log('train_losses', loss_here, self.current_epoch)
        gs = group_sparsity_universal(self.network)
        agg = {}
        for name, alive in gs.items():
            k = _stage_key(name)
            a, t = agg.get(k, (0, 0))
            agg[k] = (a + int(alive.sum()), t + alive.numel())
        row = {'epoch': self.current_epoch,
               'lr': float(self.optimizer.param_groups[0]['lr'])}
        for key in sorted(agg):
            a, t = agg[key]
            row[f'{key}/alive'] = a
            row[f'{key}/total'] = t
        row['alive_total'] = sum(a for a, _ in agg.values())
        row['total'] = sum(t for _, t in agg.values())
        csv_path = os.path.join(self.output_folder, 'sparsity_log.csv')
        new = not os.path.exists(csv_path)
        with open(csv_path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)

    def on_epoch_end(self):
        super().on_epoch_end()
        if self.local_rank == 0 and (self.current_epoch + 1) % 10 == 0:
            try:
                os.makedirs('/kaggle/working/prox_ckpts', exist_ok=True)
                shutil.copy(join(self.output_folder, 'checkpoint_latest.pth'),
                            '/kaggle/working/prox_ckpts/checkpoint_latest.pth')
            except Exception as e:
                print('[snap]', e)
        if (time.time() - KERNEL_T0) > BUDGET_H * 3600 and not SMOKE:
            self.save_checkpoint(join(self.output_folder, 'checkpoint_latest.pth'))
            print(f'[budget] стоп на эпохе {self.current_epoch}')
            raise _SessionOver

In [ ]:
# ============ ЯЧЕЙКА 6: ЗАПУСК ЭТАПА A ============
trainer = nnUNetTrainerProxKD(plans=plans, configuration=CONFIG,
                              fold=FOLD, dataset_json=dataset_json)
trainer.num_processes = NUM_DL_WORKERS
if SMOKE:
    trainer.num_epochs = 2
    trainer.num_iterations_per_epoch = 5
    trainer.num_val_iterations_per_epoch = 2
    trainer.ft_warmup = 0

trainer.initialize()
net = trainer.network
if hasattr(net, '_orig_mod'):
    net = net._orig_mod
net.load_state_dict(ckpt_distill['network_weights'], strict=True)
print('старт из дистиллята (strict) | lr =', trainer.optimizer.param_groups[0]['lr'])

try:
    trainer.run_training()
except _SessionOver:
    print('[run] бюджет-стоп')

final_ckpt = join(trainer.output_folder, 'checkpoint_final.pth')
if not os.path.isfile(final_ckpt):
    final_ckpt = join(trainer.output_folder, 'checkpoint_latest.pth')
prox_sd = torch.load(final_ckpt, map_location='cpu', weights_only=False)['network_weights']
torch.save({'network_weights': prox_sd, 'epoch': trainer.current_epoch},
           '/kaggle/working/prox_stage_a.pth')
print('этап A завершён, эпоха:', trainer.current_epoch)

In [ ]:
# ============ ЯЧЕЙКА 7: ОТЧЁТ О СПАРСИФИКАЦИИ ============
net_a = build_net(FEATS_BASE)
net_a.load_state_dict(prox_sd, strict=True)
gs0 = group_sparsity_universal(BASE)
gs1 = group_sparsity_universal(net_a)
agg0, agg1 = {}, {}
for name, alive in gs0.items():
    k = _stage_key(name)
    a, t = agg0.get(k, (0, 0))
    agg0[k] = (a + int(alive.sum()), t + alive.numel())
for name, alive in gs1.items():
    k = _stage_key(name)
    a, t = agg1.get(k, (0, 0))
    agg1[k] = (a + int(alive.sum()), t + alive.numel())
print(f"{'стадия':10s} {'было':>6s} {'стало':>6s} {'смертность':>10s}")
tot0, tot1 = 0, 0
for k in sorted(set(agg0) | set(agg1)):
    a0, t0 = agg0.get(k, (0, 0))
    a1, t1 = agg1.get(k, (0, 0))
    tot0 += t0; tot1 += a1
    print(f'{k:10s} {t0:6d} {a1:6d} {100*(1-a1/t0) if t0 else 0:9.1f}%')
print(f'ИТОГО: {tot1}/{tot0} живых ({100*tot1/tot0:.1f}%)')

In [ ]:
# ============ ЯЧЕЙКА 8: ФИЗИЧЕСКАЯ РЕЗКА ПО ALIVE ============
alive_map = group_sparsity_universal(net_a)
fK = {}
net_a = net_a.cpu()
for st in CHAIN:
    cont = net_a.encoder.stem if st == 'stem' else net_a.encoder.stages[int(st[3:])]
    blocks = list(cont.convs) if st == 'stem' else get_blocks(cont)
    first_conv = _units_of(blocks[0])[0][1]
    name = None
    for mod_name, mod in net_a.named_modules():
        if mod is first_conv:
            name = mod_name
            break
    assert name and name in alive_map, f'нет alive для {st}'
    fK[st] = torch.nonzero(alive_map[name]).flatten().cpu()
    print(f'  {st}: {len(fK[st])}/{first_conv.out_channels} alive')

# пары с обязательным равенством
for a, b in (('stem', 'enc0'), ('enc4', 'enc5')):
    ka, kb = fK[a], fK[b]
    if len(ka) == 0 or len(kb) == 0:
        print(f'  пара {a}<->{b}: {b} мёртв — {a} по своему alive')
        continue
    common = torch.tensor(sorted(set(ka.tolist()) & set(kb.tolist())), dtype=torch.long)
    fK[a] = fK[b] = common
    print(f'  пара {a}<->{b}: {len(common)} общих')

# encoder резка
prev_keep = None
for st in CHAIN:
    keep = fK[st]
    if keep is None or len(keep) == 0:
        if prev_keep is not None and len(prev_keep) > 0:
            keep = prev_keep
            fK[st] = keep
            print(f'  {st}: 0 alive — режем по prev ({len(keep)}), веса нулевые')
        else:
            fK[st] = None
            prev_keep = None
            print(f'  {st}: пусто — не режем')
            continue
    cont = net_a.encoder.stem if st == 'stem' else net_a.encoder.stages[int(st[3:])]
    blocks = list(cont.convs) if st == 'stem' else get_blocks(cont)
    for bi, b in enumerate(blocks):
        for tag, conv, norm in _units_of(b):
            _slice_out(conv, keep, norm)
            if st == 'stem':
                if bi > 0:
                    _slice_in(conv, keep)
            else:
                if tag == 'conv2' or bi > 0:
                    _slice_in(conv, keep)
                elif tag in ('conv1', 'skip') and prev_keep is not None:
                    _slice_in(conv, prev_keep)
        _force_norm_sync(b, len(keep), keep)
    prev_keep = keep

# decoder
for k in range(len(net_a.decoder.transpconvs)):
    Ks = fK[CHAIN[5 - k]]
    tc = net_a.decoder.transpconvs[k]
    if Ks is None or len(Ks) == 0:
        continue
    if k == 0:
        Kb = fK['enc5']
        if Kb is not None and len(Kb) > 0:
            tc.weight.data = tc.weight.data.index_select(0, Kb).contiguous()
            tc.in_channels = len(Kb)
    else:
        Kprev = fK[CHAIN[5 - (k - 1)]]
        if Kprev is not None and len(Kprev) > 0:
            tc.weight.data = tc.weight.data.index_select(0, Kprev).contiguous()
            tc.in_channels = len(Kprev)
    tc.weight.data = tc.weight.data.index_select(1, Ks).contiguous()
    if tc.bias is not None:
        tc.bias.data = tc.bias.data.index_select(0, Ks).contiguous()
    tc.out_channels = len(Ks)
    unit = net_a.decoder.stages[k].convs[0]
    conv = getattr(unit, 'conv', unit)
    norm = getattr(unit, 'norm', None)
    w = conv.weight.data
    half = w.shape[1] // 2
    if half == len(Ks):
        conv.in_channels = 2 * len(Ks)
    else:
        up_part = w[:, :half].index_select(1, Ks.to(w.device)).contiguous()
        skip_part = w[:, half:].index_select(1, Ks.to(w.device)).contiguous()
        parts = [skip_part, up_part] if DECODER_CAT_SKIP_FIRST else [up_part, skip_part]
        conv.weight.data = torch.cat(parts, dim=1).contiguous()
        conv.in_channels = conv.weight.data.shape[1]
    _slice_out(conv, Ks, norm)
    _force_norm_sync(unit, len(Ks), Ks)
    _slice_in(net_a.decoder.seg_layers[k], Ks)

_normalize_skips(net_a.encoder, CHAIN)
_sync_duplicate_encoder(net_a, fK, CHAIN)

feats7 = [net_a.encoder.stem.convs[0].conv.out_channels]
for st in net_a.encoder.stages:
    last = get_blocks(st)[-1]
    for tag in ('conv2', 'conv1', 'conv'):
        b = getattr(last, tag, None)
        if b is not None:
            feats7.append(getattr(b, 'conv', b).out_channels)
            break
print('feats7 (резаные):', feats7)
print('параметров: %.1fM' % (sum(p.numel() for p in net_a.parameters())/1e6))
sd_cut = {k: v.cpu() for k, v in net_a.state_dict().items()}
torch.save({'network_weights': sd_cut, 'feats7': feats7},
           '/kaggle/working/prox_cut_model.pth')

In [ ]:
# ============ ЯЧЕЙКА 9: МИКРОТЕСТ РЕЗКИ ============
gc.collect(); torch.cuda.empty_cache()
fp = fix_monotonic(feats7[1:], kw['strides'])
plans_cut = load_json(join(ds_work, 'nnUNetResEncUNetMPlans.json'))
plans_cut['configurations'][CONFIG]['architecture']['arch_kwargs']['features_per_stage'] = fp
plans_cut['continue_training'] = False
save_json(plans_cut, join(ds_work, 'nnUNetResEncUNetMPlans.json'))

test_net = build_net(feats7[1:])
ref = test_net.state_dict()
cur = {k: v.cpu() for k, v in net_a.state_dict().items()}
mismatch = [k for k in ref if k in cur and tuple(ref[k].shape) != tuple(cur[k].shape)]
missing = [k for k in ref if k not in cur]
extra = [k for k in cur if k not in ref]
print(f'mismatch: {len(mismatch)} | missing: {len(missing)} | extra: {len(extra)}')
assert not mismatch and not missing and not extra, 'формы не сошлись'

net_a = net_a.cuda()
with torch.no_grad():
    y = net_a(torch.randn(1, IN_CH, *PATCH).cuda())
print('forward OK:', tuple(y.shape))
net_a = net_a.cpu()
gc.collect(); torch.cuda.empty_cache()
print('МИКРОТЕСТ ПРОЙДЕН')

In [ ]:
# ============ ЯЧЕЙКА 10: RECOVERY-ТРЕНЕР (этап C) ============
class nnUNetTrainerRecoveryProx(nnUNetTrainer):
    def __init__(self, plans, configuration, fold, dataset_json,
                 device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = RECOVERY_EPOCHS if not SMOKE else 2
        self.num_val_iterations_per_epoch = 10 if not SMOKE else 2
        self.save_every = 5
        self._t0 = None
        self.kd_teachers = TEACHERS
        self.kd_teacher_device = TEACHER_DEVICE
        if SMOKE:
            self.num_iterations_per_epoch = 5

    def configure_optimizers(self, lr_init=None, new_weight_decay=None):
        optimizer = torch.optim.SGD(self.network.parameters(), RECOVERY_LR,
                                    weight_decay=WEIGHT_DECAY,
                                    momentum=RECOVERY_MOMENTUM, nesterov=True)
        scheduler = torch.optim.lr_scheduler.PolynomialLR(
            optimizer, total_iters=self.num_epochs, power=1.0)
        self.optimizer, self.lr_scheduler = optimizer, scheduler
        return optimizer, scheduler

    def train_step(self, batch):
        data = batch['data'].to(self.device, non_blocking=True)
        target = batch['target']
        if isinstance(target, list):
            target = [t.to(self.device, non_blocking=True) for t in target]
        else:
            target = target.to(self.device, non_blocking=True)
        self.optimizer.zero_grad(set_to_none=True)
        with torch.autocast(self.device.type, enabled=self.device.type == 'cuda'):
            output = self.network(data)
            shapes = [tuple(o.shape[2:]) for o in output]
            tp = teacher_probs_fn(data)
            t_list = []
            for shp in shapes:
                if tuple(tp.shape[2:]) == tuple(shp):
                    t_list.append(tp)
                else:
                    t_list.append(torch.nn.functional.interpolate(
                        tp, size=shp, mode='area').clamp_min(1e-6))
            n = len(output)
            ws = [1.0 / (2 ** i) for i in range(n)]
            ws = [w / sum(ws) for w in ws]
            loss = sum(w * (soft_ce(o, t) + soft_dice(o, t))
                       for w, o, t in zip(ws, output, t_list))
        if self.grad_scaler is not None:
            self.grad_scaler.scale(loss).backward()
            self.grad_scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.grad_scaler.step(self.optimizer)
            self.grad_scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.optimizer.step()
        return {'loss': loss.detach().cpu().numpy()}

    def on_epoch_end(self):
        super().on_epoch_end()
        if (time.time() - KERNEL_T0) > BUDGET_H * 3600 and not SMOKE:
            self.save_checkpoint(join(self.output_folder, 'checkpoint_latest.pth'))
            print(f'[budget] стоп на эпохе {self.current_epoch}')
            raise _SessionOver

trainer_rec = nnUNetTrainerRecoveryProx(plans=plans_cut, configuration=CONFIG,
                                        fold=FOLD, dataset_json=dataset_json)
trainer_rec.num_processes = NUM_DL_WORKERS
trainer_rec.initialize()
net_r = trainer_rec.network
if hasattr(net_r, '_orig_mod'):
    net_r = net_r._orig_mod
net_r.load_state_dict({k: v.cuda() for k, v in sd_cut.items()}, strict=True)
print('recovery стартует из резаной модели (strict)')

In [ ]:
# ============ ЯЧЕЙКА 11: ЗАПУСК ЭТАПА C (recovery) ============
try:
    trainer_rec.run_training()
except _SessionOver:
    print('[run] бюджет-стоп')

rec_ckpt = join(trainer_rec.output_folder, 'checkpoint_final.pth')
if not os.path.isfile(rec_ckpt):
    rec_ckpt = join(trainer_rec.output_folder, 'checkpoint_latest.pth')
rec_sd = torch.load(rec_ckpt, map_location='cpu', weights_only=False)['network_weights']
torch.save({'network_weights': rec_sd, 'feats7': feats7},
           '/kaggle/working/prox_rec_model.pth')
print('recovery завершён, эпоха:', trainer_rec.current_epoch)

In [ ]:
# ============ ЯЧЕЙКА 12: ВАЛИДАЦИЯ RECOVERY ============
import subprocess

def make_stub(network_weights, net_ref, src_ckpt):
    dummy_opt = torch.optim.SGD(net_ref.parameters(), lr=1e-3)
    return {'network_weights': network_weights,
            'optimizer_state': dummy_opt.state_dict(),
            'grad_scaler_state': None,
            'logging': src_ckpt.get('logging'),
            '_best_ema': src_ckpt.get('_best_ema'),
            'current_epoch': src_ckpt.get('current_epoch', 0),
            'init_args': src_ckpt.get('init_args'),
            'trainer_name': 'nnUNetTrainer',
            'inference_allowed_mirroring_axes': None}  # БЕЗ TTA-зеркал!

dst = (f'/kaggle/working/nnUNet_results/{DATASET_NAME}/'
       f'nnUNetTrainer__nnUNetResEncUNetMPlans__{CONFIG}/fold_{FOLD}')
os.makedirs(dst, exist_ok=True)
torch.save(make_stub(rec_sd, net_a, ckpt_distill), join(dst, 'checkpoint_final.pth'))
if os.path.exists(join(dst, 'validation')):
    shutil.rmtree(join(dst, 'validation'))
env = dict(os.environ)
env['nnUNet_preprocessed'] = ds_work_root
r = subprocess.run(['nnUNetv2_train', str(DATASET_ID), CONFIG, str(FOLD),
                    '-tr', 'nnUNetTrainer', '-p', 'nnUNetResEncUNetMPlans', '--val'],
                   capture_output=True, text=True, timeout=7200, env=env)
s_path = join(dst, 'validation/summary.json')
assert os.path.exists(s_path), f'валидация не дала summary\n{r.stdout[-800:]}\n{r.stderr[-800:]}'
s_rec = json.load(open(s_path))
fm = s_rec.get('foreground_mean') or {}
d_rec = fm.get('Dice', fm.get('DICE', -1))
print(f'\n=== ИТОГ: recovery mean Dice = {d_rec:.4f} ===')
print(f'гейт 0.860: {"ПРОЙДЕН" if d_rec >= 0.86 else "ПРОВАЛЕН"}')
mean_blk = s_rec.get('mean') or {}
organ_labels = ["BrachialPlexus_R", "BrachialPlexus_L", "SpinalCanal", "SpinalCord",
                "Trachea", "Esophagus", "Thyroid", "A_Carotid_R", "A_Carotid_L",
                "Larynx", "PharynxConst_I", "PharynxConst_M", "PharynxConst_S",
                "OpticNerve_R", "OpticNerve_L", "Brain", "Submandibular_R",
                "Submandibular_L", "Parotid_R", "Parotid_L", "Lips", "Tongue",
                "Mandible", "Eye_R", "Lens_R", "Eye_L", "Lens_L", "TMJoint_R",
                "TMJoint_L", "Cochlea_R", "Cochlea_L"]
for k in sorted(mean_blk, key=lambda x: int(x)):
    if k.isdigit() and int(k)-1 < len(organ_labels):
        print(f"  {organ_labels[int(k)-1]:18s} {mean_blk[k]['Dice']:.4f}")
os.makedirs('/kaggle/working/validations', exist_ok=True)
shutil.copy(s_path, '/kaggle/working/validations/RECOVERY_summary.json')

In [ ]:
# ============ ЯЧЕЙКА 13: INT8-КВАНТИЗАЦИЯ (NNCF PTQ, subprocess) ============
# NNCF выполняется отдельным процессом: nbclient на Kaggle рвёт долгие ячейки по
# iopub-таймауту; stdout раннера стримится — прогресс живой, таймаутов нет.
import blosc2
import onnx  # noqa: F401 — нужен torch.onnx.export
import openvino as ov

net_e = build_net(feats7[1:])
net_e.load_state_dict(rec_sd, strict=True)
net_e.eval().cuda()

dummy = torch.randn(1, IN_CH, *PATCH).cuda()
onnx_path = '/kaggle/working/prox_rec.onnx'
torch.onnx.export(net_e, (dummy,), onnx_path, export_params=True, opset_version=11,
                  input_names=['x'], output_names=['y'], do_constant_folding=True)
# torch>=2.9 может вынести веса в external data — переэкспорт single-file
if os.path.isfile(onnx_path + '.data'):
    os.remove(onnx_path); os.remove(onnx_path + '.data')
    try:  # dynamo-экспортёр
        torch.onnx.export(net_e, (dummy,), onnx_path, export_params=True, opset_version=11,
                          input_names=['x'], output_names=['y'], do_constant_folding=True,
                          external_data=False, dynamo=True)
    except TypeError:  # legacy-экспортёр
        torch.onnx.export(net_e, (dummy,), onnx_path, export_params=True, opset_version=11,
                          input_names=['x'], output_names=['y'], do_constant_folding=True,
                          save_as_external_data=False)
assert not os.path.isfile(onnx_path + '.data'), 'веса ушли в external data'
assert os.path.getsize(onnx_path) / 1e6 > 100, 'ONNX слишком мал — веса потеряны'
print('ONNX: %d МБ' % (os.path.getsize(onnx_path) / 1e6))
net_e = net_e.cpu(); gc.collect(); torch.cuda.empty_cache()

core = ov.Core()
ov_model = ov.convert_model(onnx_path)
ov.save_model(ov_model, '/kaggle/working/prox_rec_fp32.xml')
print('FP32 IR сохранён')

# калибровочные патчи из предобработанных данных (b2nd)
data_id_c = cfg.get('data_identifier', 'nnUNetPlans_3d_fullres')
DATA_DIR_C = join(ds_work, data_id_c)
case_files = [f for f in sorted(glob.glob(join(DATA_DIR_C, '*.b2nd')))
              if not f.endswith('_seg.b2nd')]
random.shuffle(case_files)
K_CAL = 24 if SMOKE else 100
patches = []
for cf in case_files:
    arr = np.asarray(blosc2.open(cf)[...])
    if arr.ndim == 4:
        arr = arr[0] if arr.shape[0] == 1 else arr
    if arr.ndim == 3:
        arr = arr[None]
    for _ in range(8):
        off = [random.randint(0, max(arr.shape[i + 1] - PATCH[i], 0)) for i in range(3)]
        patch_x = arr[:, off[0]:off[0] + PATCH[0], off[1]:off[1] + PATCH[1], off[2]:off[2] + PATCH[2]]
        t = torch.from_numpy(np.ascontiguousarray(patch_x, dtype=np.float32))
        t = (t - t.mean()) / (t.std() + 1e-8)
        patches.append(t.numpy())
        if len(patches) >= K_CAL:
            break
    if len(patches) >= K_CAL:
        break
print('калибровочных патчей:', len(patches))
np.save('/kaggle/working/calibration_patches.npy', np.stack(patches).astype(np.float32))

runner_code = chr(10).join([
    "import functools, sys, time",
    "print = functools.partial(print, flush=True)",
    "import numpy as np, os",
    "os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')",
    "sys.modules.setdefault('tensorflow', None)",
    "sys.modules.setdefault('tensorboard', None)",
    "import nncf, openvino as ov",
    "K, xml_fp32, xml_out, calib_npy = sys.argv[1:5]",
    "K = int(K)",
    "calib = np.load(calib_npy)",
    "K = min(K, len(calib))",
    "print(f'runner: NNCF на {K} патчах из {len(calib)}')",
    "core = ov.Core()",
    "model = core.read_model(xml_fp32)",
    "def gen():",
    "    for i in range(K):",
    "        print(f'[NNCF] патч {i+1}/{K}', flush=True)",
    "        yield calib[i][np.newaxis]",
    "t0 = time.time()",
    "q = nncf.quantize(model, nncf.Dataset(gen()),",
    "                  preset=nncf.QuantizationPreset.MIXED, subset_size=K)",
    "print(f'runner: NNCF {time.time()-t0:.0f} с')",
    "ov.serialize(q, xml_out)",
    "print('runner: сериализовано', xml_out)",
])
with open('/kaggle/working/runner_quant.py', 'w') as f:
    f.write(runner_code)
proc = subprocess.Popen([sys.executable, '/kaggle/working/runner_quant.py',
                         str(len(patches)), '/kaggle/working/prox_rec_fp32.xml',
                         '/kaggle/working/prox_rec_int8.xml',
                         '/kaggle/working/calibration_patches.npy'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for ln in proc.stdout:
    print(ln.rstrip())
proc.wait()
assert proc.returncode == 0 and os.path.isfile('/kaggle/working/prox_rec_int8.xml'), \
    f'NNCF-раннер упал (код {proc.returncode}) — см. вывод выше'

# argmax-гейт fp32 vs int8 на CPU-OpenVINO
fp32_c = ov.Core().compile_model('/kaggle/working/prox_rec_fp32.xml', 'CPU')
int8_c = ov.Core().compile_model('/kaggle/working/prox_rec_int8.xml', 'CPU')
agree = []
for t in patches[:8]:
    a = fp32_c(t[None])[fp32_c.output(0)][0].argmax(0)
    b = int8_c(t[None])[int8_c.output(0)][0].argmax(0)
    agree.append(float((a == b).mean()))
mean_agree = float(np.mean(agree))
print(f'argmax-совпадение: {mean_agree:.4f} (гейт 0.9: {"OK" if mean_agree >= 0.9 else "FAIL"})')

In [ ]:
# ============ ЯЧЕЙКА 14: ИТОГОВЫЙ ОТЧЁТ ============
report = {
    'config': {'group_lambda': GROUP_LAMBDA, 'indiv_lambda': INDIV_LAMBDA,
               'momentum': MOMENTUM, 'lr': LR_INIT, 'warmup': WARMUP_EPOCHS,
               'epochs': NUM_EPOCHS, 'recovery_epochs': RECOVERY_EPOCHS},
    'stage_a': {'epochs_done': trainer.current_epoch},
    'stage_b': {'feats7': feats7,
                'params_M': round(sum(p.numel() for p in net_a.parameters())/1e6, 1)},
    'stage_c': {'epochs_done': trainer_rec.current_epoch},
    'validation': {'mean_dice': d_rec, 'gate_0.86': d_rec >= 0.86},
    'int8': {'argmax_agree': mean_agree},
}
json.dump(report, open('/kaggle/working/report.json', 'w'), indent=1)
print(json.dumps(report, indent=1, ensure_ascii=False))
print(f'\nПОЛНОЕ ВРЕМЯ: {(time.time() - KERNEL_T0)/3600:.1f} ч')